# Path-Signature Volatility Forecasting Experiments

This notebook is intentionally **results-first**. It suppresses the long feature-generation logs and only displays compact experiment tables.

It runs the same experiment sequence for all five requested horizons:

- **5 min**
- **1 hour**
- **1 day**
- **7 day**
- **30 day**

The experiment sequence is:

1. **Depth 3, full path:** HAR-RV-L, XGBoost (stats), Signature Ridge, XGBoost (signature).
2. **Depth 3 vs depth 4:** compare the two signature models. (Depth 4 is left disabled -- it crashes on this machine.)
3. **No augmentation vs time vs lead-lag + time:** compare the two signature models.
4. **Path dimensions:** compare price, price + activity, and the full source-compatible path.
5. **Full model comparison (1 day / 7 day / 30 day only):** HAR-RV-L, GARCH(1,1), XGBoost (stats/signature), Signature Ridge, MLP (stats/stats+signature), and a signature-LSTM, evaluated with a walk-forward backtest and ranked by RMSE with a Diebold-Mariano test against the best model. EGARCH is temporarily disabled.

For Binance, the full path is **price + activity + flow**. Yahoo does not contain signed trade flow, so the daily Yahoo experiments use **price + activity + range**, matching the original daily specification.

**Baseline change:** the old "Random Forest on stats" baseline is now XGBoost on the same stats features, so the *only* difference between the "stats" and "signature" baselines is the feature set, not the model family.

**HAR change:** HAR now follows the standard crypto HAR-RV formulation -- RV(1)/RV(7)/RV(30) rolling means of a daily variance proxy, fit in log-variance space with a leverage term (HAR-RV-L), with a direct per-horizon regression rather than one HAR shape reused across horizons.

**New dependencies:** `pip install arch torch statsmodels` (on top of the existing requirements) before running this notebook.


## 1. Imports and quiet output

In [2]:
from pathlib import Path
from dataclasses import replace
from collections import OrderedDict
import gc
import logging
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from forecast_data import (
    SPECS,
    build_dataset,
    binance_to_bars,
    chronological_split,
    duration_steps,
    load_or_download_data,
)

from models_and_metrics import (
    EPS,
    WalkForwardResult,
    compare_models,
    evaluate_garch,
    evaluate_har_log,
    evaluate_model,
    fit_and_evaluate_standard_models,
    fit_mlp,
    fit_signature_lstm,
    fit_signature_ridge,
    fit_xgb,
    walk_forward_folds,
)

# Keep notebook output focused on final result tables.
# Change WARNING -> INFO if you want detailed progress.
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("forecast_data").setLevel(logging.WARNING)
logging.getLogger("models_and_metrics").setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## 2. Configuration
Change dates and experiment settings here.

In [3]:
DATA_ROOT = Path("data")
OUTPUT_ROOT = Path("outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# =================================================
# DATA RANGES
# =================================================

# Binance is used for 5-minute and 1-hour forecasts.
BINANCE_SYMBOL = "BTCUSDT"
BINANCE_START = "2026-08-01"
BINANCE_END = "2026-08-31"       # inclusive

# Yahoo is used for 1-day, 7-day, and 30-day forecasts.
YAHOO_TICKER = "BTC-USD"
YAHOO_START = "2020-01-01"
YAHOO_END = "2026-09-16"         # inclusive

# Existing cached files are reused unless this is True.
FORCE_DOWNLOAD = False

# Final chronological holdout for Experiments 1-4 (the signature ablations).
TEST_FRACTION = 0.20

# Main metric shown in the compact ablation tables. Only RMSE is tracked now
# (MAE/QLIKE were dropped); Experiment 5 additionally reports Diebold-Mariano
# significance against the best model per horizon.
PRIMARY_METRIC = "RMSE"

# All requested horizons.
HORIZONS = ["5m", "1h", "1d", "7d", "30d"]

# Experiment 5 (full model comparison incl. GARCH/MLP/LSTM) only makes
# sense for the daily-bar horizons -- GARCH and the 5-day-window LSTM are
# built around daily variance, not 5s/1min bars.
DAILY_HORIZONS = ["1d", "7d", "30d"]

HORIZON_LABELS = {
    "5m": "5 min",
    "1h": "1 hour",
    "1d": "1 day",
    "7d": "7 day",
    "30d": "30 day",
}

# Keep XGBoost enabled in the signature ablations (Experiments 2-4).
INCLUDE_XGB = True

# Walk-forward folds for Experiment 5 (expanding window, via TimeSeriesSplit).
N_SPLITS = 4

# MLP weight decay per horizon -- heavier regularization for the longer
# horizons, since fewer independent samples are available there.
WEIGHT_DECAY = {"1d": 1e-4, "7d": 5e-4, "30d": 1e-3}

# Memory: at most this many raw-tick / bars / feature-dataset objects are
# kept alive at once (LRU -- oldest evicted first). Lower this further if
# the kernel is still running out of memory; raise it only if you have
# RAM to spare and want to avoid recomputation across experiments.
MAX_CACHED_OBJECTS = 1


## 3. Helper functions

These helpers:

1. load each raw source only once,
2. cache each unique forecast-dataset specification, and
3. return compact result DataFrames instead of printing feature-generation details.

In [4]:
class BoundedCache(OrderedDict):
    """
    Small LRU cache so we never hold more than `max_size` big objects (raw
    ticks / bars / feature datasets) in memory at once. get_or_build reuses a
    cached value if present, otherwise builds it, stores it, and evicts the
    least-recently-used entry once the cache is over capacity.
    """
    def __init__(self, max_size=MAX_CACHED_OBJECTS):
        super().__init__()
        self.max_size = max_size

    def get_or_build(self, key, builder):
        if key in self:
            self.move_to_end(key)
            return self[key]
        value = builder()
        self[key] = value
        self.move_to_end(key)
        while len(self) > self.max_size:
            self.popitem(last=False)
            gc.collect()
        return value


RAW_CACHE = BoundedCache()
BARS_CACHE = BoundedCache()
DATASET_CACHE = BoundedCache()


def horizon_source(horizon):
    if horizon in {"5m", "1h"}:
        return {
            "source": "binance",
            "start": BINANCE_START,
            "end": BINANCE_END,
            "symbol": BINANCE_SYMBOL,
        }

    return {
        "source": "yahoo",
        "start": YAHOO_START,
        "end": YAHOO_END,
        "ticker": YAHOO_TICKER,
    }


def get_raw_data(horizon):
    cfg = horizon_source(horizon)
    source = cfg["source"]

    if source == "binance":
        cache_key = (
            "binance",
            BINANCE_SYMBOL,
            BINANCE_START,
            BINANCE_END,
        )
    else:
        cache_key = (
            "yahoo",
            YAHOO_TICKER,
            YAHOO_START,
            YAHOO_END,
        )

    return RAW_CACHE.get_or_build(cache_key, lambda: load_or_download_data(
        source=source,
        start=cfg["start"],
        end=cfg["end"],
        symbol=cfg.get("symbol", BINANCE_SYMBOL),
        ticker=cfg.get("ticker", YAHOO_TICKER),
        data_root=DATA_ROOT,
        force_download=FORCE_DOWNLOAD,
    ))


def get_bars(horizon):
    def build():
        raw = get_raw_data(horizon)
        spec = SPECS[horizon]
        if horizon in {"5m", "1h"}:
            bars = binance_to_bars(raw, spec.bar_freq)
            # The raw trade table is much larger than the resampled bars.
            # Do not retain both while signature features/models are built.
            RAW_CACHE.clear()
            del raw
            gc.collect()
            return bars
        return raw.copy()

    return BARS_CACHE.get_or_build(horizon, build)


def get_returns(horizon):
    """Daily log-returns for GARCH/EGARCH -- reuses the same cached bars."""
    return np.log(get_bars(horizon)["close"]).diff().dropna()


def spec_key(horizon, spec):
    return (
        horizon,
        spec.depth,
        spec.lead_lag,
        spec.time_aug,
        tuple(spec.dims),
        spec.path_points,
        spec.sample_stride,
        spec.variance_method,
    )


def get_dataset(horizon, spec=None, include_lstm_seq=False):
    """
    One dataset object per (horizon, spec, include_lstm_seq) -- HAR/stats/
    signature features are always included; the LSTM's sub-window signature
    sequence (X_seq) is only computed when include_lstm_seq=True, so
    ablation experiments that don't need it never pay for it or hold it.
    """
    spec = spec or SPECS[horizon]
    key = spec_key(horizon, spec) + (include_lstm_seq,)
    return DATASET_CACHE.get_or_build(
        key, lambda: build_dataset(get_bars(horizon), spec, include_lstm_seq=include_lstm_seq)
    )


def signature_model_results(dataset):
    train_idx, test_idx = chronological_split(
        dataset,
        test_fraction=TEST_FRACTION,
    )

    y_train = dataset.y[train_idx]
    y_test = dataset.y[test_idx]

    results = {}

    ridge = fit_signature_ridge(
        dataset.X_sig[train_idx],
        y_train,
    )
    results["Signature Ridge"] = evaluate_model(
        "Signature Ridge",
        ridge,
        dataset.X_sig[test_idx],
        y_test,
    )

    if INCLUDE_XGB:
        xgb = fit_xgb(
            dataset.X_sig[train_idx],
            y_train,
        )
        results["Signature XGBoost"] = evaluate_model(
            "Signature XGBoost",
            xgb,
            dataset.X_sig[test_idx],
            y_test,
        )

    return results


def result_rows(results, horizon, experiment, setting):
    rows = []

    for model_name, result in results.items():
        rows.append({
            "Horizon": HORIZON_LABELS[horizon],
            "HorizonKey": horizon,
            "Experiment": experiment,
            "Setting": setting,
            "Model": model_name,
            **result.metrics,
        })

    return rows


def ordered_results(df):
    out = df.copy()
    out["Horizon"] = pd.Categorical(
        out["Horizon"],
        categories=[HORIZON_LABELS[h] for h in HORIZONS],
        ordered=True,
    )
    return out.sort_values(["Horizon", "RMSE"])


def show_baseline_table(df, metric=PRIMARY_METRIC):
    table = (
        df.pivot(
            index="Horizon",
            columns="Model",
            values=metric,
        )
        .reindex([HORIZON_LABELS[h] for h in HORIZONS])
    )

    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"Baseline -- {metric} (bold = lowest within horizon)")
    )

    return table


def show_ablation_table(df, metric=PRIMARY_METRIC):
    table = df.pivot_table(
        index=["Horizon", "Model"],
        columns="Setting",
        values=metric,
        aggfunc="first",
    )

    horizon_order = [HORIZON_LABELS[h] for h in HORIZONS]
    order_map = {h: i for i, h in enumerate(horizon_order)}

    temp = table.reset_index()
    temp["_order"] = temp["Horizon"].map(order_map)
    temp = temp.sort_values(["_order", "Model"]).drop(columns="_order")
    table = temp.set_index(["Horizon", "Model"])

    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"{metric} comparison (bold = lowest across settings in each row)")
    )

    return table


def best_dims_for_horizon(horizon, dimension_df):
    """Read Experiment 4's own ablation results and return the path-dimension
    tuple with the lowest RMSE for this horizon -- reused directly instead of
    re-running the dimension search inside Experiment 5."""
    sub = dimension_df[dimension_df["HorizonKey"] == horizon]
    best_setting = sub.loc[sub["RMSE"].idxmin(), "Setting"]
    return dimension_variants(horizon)[best_setting]


## 4. Experiment 1 — Baseline: HAR-RV-L, XGBoost (stats), Signature Ridge, XGBoost (signature)

This is the main baseline.

For each horizon:

- **HAR-RV-L** uses log-RV(1)/RV(7)/RV(30) + a leverage term (see the intro cell).
- **XGBoost (stats)** uses the same summary-statistic features the old Random Forest baseline used -- same feature set, but now the same model family as the signature baseline below, for an apples-to-apples comparison.
- **Signature Ridge** uses depth-3 signatures.
- **XGBoost (signature)** uses depth-3 signatures.

The baseline signature path uses both **lead-lag and time augmentation**.

- 5 min / 1 hour: `price + activity + flow`
- 1 day / 7 day / 30 day: `price + activity + range`


In [5]:
baseline_rows = []

for horizon in HORIZONS:
    spec = replace(
        SPECS[horizon],
        depth=3,
        lead_lag=True,
        time_aug=True,
    )

    dataset = get_dataset(horizon, spec)
    train_idx, test_idx = chronological_split(
        dataset,
        test_fraction=TEST_FRACTION,
    )
    horizon_steps = duration_steps(spec.horizon, spec.bar_freq)

    results = fit_and_evaluate_standard_models(
        dataset,
        train_idx,
        test_idx,
        horizon_steps,
    )

    baseline_rows.extend(
        result_rows(
            results,
            horizon=horizon,
            experiment="Baseline",
            setting="Depth 3 | full path | lead-lag + time",
        )
    )

    # Only the compact metric rows are needed later. Keeping datasets and
    # fitted model objects across horizons is what caused the kernel crash.
    del results, dataset, train_idx, test_idx
    DATASET_CACHE.clear()
    gc.collect()

baseline_df = ordered_results(pd.DataFrame(baseline_rows))
baseline_table = show_baseline_table(baseline_df)


Model,HAR-RV-L,Signature Ridge,XGBoost (signature),XGBoost (stats)
Horizon,,,,
5 min,0.000612236,0.000466614,0.000424682,0.000418914
1 hour,0.00131488,0.00189911,0.00151339,0.00166057
1 day,0.0106404,0.0095585,0.0100768,0.0123532
7 day,0.0224496,0.0191543,0.0236441,0.0398524
30 day,0.0480183,0.0399165,0.0457443,0.0795053


### Note

RMSE is the only metric tracked per model now (MAE/QLIKE were dropped). Experiment 5 below adds a Diebold-Mariano significance test on top of RMSE for the full model comparison.


In [6]:
# (nothing to run here anymore -- kept as a placeholder cell)


## 5. Experiment 2 — Depth 3 vs depth 4

Everything except signature depth is held fixed:

- full source-compatible path dimensions
- lead-lag augmentation
- time augmentation

Only the two signature models are shown because HAR and Random Forest are unaffected by signature depth.

In [7]:
# depth_rows = []

# for horizon in HORIZONS:
#     for depth in [3, 4]:
#         spec = replace(
#             SPECS[horizon],
#             depth=depth,
#             lead_lag=True,
#             time_aug=True,
#         )

#         dataset = get_dataset(horizon, spec)
#         results = signature_model_results(dataset)

#         depth_rows.extend(
#             result_rows(
#                 results,
#                 horizon=horizon,
#                 experiment="Depth",
#                 setting=f"Depth {depth}",
#             )
#         )

# depth_df = ordered_results(pd.DataFrame(depth_rows))
# depth_table = show_ablation_table(depth_df)

## 6. Experiment 3 — Augmentation

Depth is fixed at 3 and the full source-compatible dimensions are used.

The three paths are:

1. **None** — raw path only
2. **Time** — time augmentation only
3. **Lead-lag + time** — lead-lag transform followed by time augmentation

In [8]:
AUGMENTATIONS = {
    "None": {
        "lead_lag": False,
        "time_aug": False,
    },
    "Time": {
        "lead_lag": False,
        "time_aug": True,
    },
    "Lead-lag + time": {
        "lead_lag": True,
        "time_aug": True,
    },
}

augmentation_rows = []

for horizon in HORIZONS:
    for setting, kwargs in AUGMENTATIONS.items():
        spec = replace(
            SPECS[horizon],
            depth=3,
            lead_lag=kwargs["lead_lag"],
            time_aug=kwargs["time_aug"],
        )

        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)

        augmentation_rows.extend(
            result_rows(
                results,
                horizon=horizon,
                experiment="Augmentation",
                setting=setting,
            )
        )

augmentation_df = ordered_results(pd.DataFrame(augmentation_rows))
augmentation_table = show_ablation_table(augmentation_df)

## 7. Experiment 4 — Path dimensions

Depth is fixed at 3 and **lead-lag + time augmentation** is used throughout.

### Binance: 5 min and 1 hour
- Price
- Price + activity
- Price + activity + flow

### Yahoo: 1 day, 7 day, 30 day
Yahoo OHLCV does **not** have signed buyer/seller trade flow. To keep the experiment valid with the existing Yahoo source, the third dimension is the daily range path from the original specification:

- Price
- Price + activity
- Price + activity + range

If you later switch the daily horizons to a source containing signed trade flow, change the Yahoo full-dimension tuple to `("price", "activity", "flow")`.

In [9]:
def dimension_variants(horizon):
    if horizon in {"5m", "1h"}:
        return {
            "Price": ("price",),
            "Price + activity": ("price", "activity"),
            "Price + activity + flow": ("price", "activity", "flow"),
        }

    return {
        "Price": ("price",),
        "Price + activity": ("price", "activity"),
        "Price + activity + range": ("price", "activity", "range"),
    }


dimension_rows = []

for horizon in HORIZONS:
    for setting, dims in dimension_variants(horizon).items():
        spec = replace(
            SPECS[horizon],
            depth=3,
            lead_lag=True,
            time_aug=True,
            dims=dims,
        )

        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)

        dimension_rows.extend(
            result_rows(
                results,
                horizon=horizon,
                experiment="Dimensions",
                setting=setting,
            )
        )

dimension_df = ordered_results(pd.DataFrame(dimension_rows))
dimension_table = show_ablation_table(dimension_df)

## 8. Experiment 5 -- full model comparison (1 day / 7 day / 30 day)

Only the daily-bar horizons run here -- GARCH and the 5-day-window signature-LSTM are built around a daily variance proxy, not 5s/1min bars. EGARCH is temporarily disabled.

For each horizon:

- Path dimensions are fixed to whichever setting had the lowest RMSE for that horizon in Experiment 4 above.
- Every model is evaluated on the same **walk-forward (expanding-window) backtest** (`N_SPLITS` folds), not a single train/test split, so the comparison isn't just one lucky/unlucky holdout.
- Models: **HAR-RV-L**, **GARCH(1,1)**, **XGBoost (stats)**, **XGBoost (signature)**, **Signature Ridge**, **MLP (stats)**, **MLP (stats + signature)**, **Signature LSTM**. EGARCH is commented out for now.
- The table is ranked by RMSE, with a **Diebold-Mariano test** of every model against the current best -- a low p-value means that model is *significantly* worse than the best, not just numerically worse.

**Memory:** each horizon's dataset (all four feature sets, computed in a single pass) is cleared before moving to the next horizon -- only one horizon's data is ever alive at a time here.


In [10]:
def run_full_comparison(horizon, dimension_df, n_splits=N_SPLITS):
    spec = replace(SPECS[horizon], depth=3, lead_lag=True, time_aug=True,
                    dims=best_dims_for_horizon(horizon, dimension_df))
    # include_lstm_seq=True computes X_har/X_stats/X_sig/X_seq together in one
    # pass -- there's only ever one dataset object per horizon here, not two.
    dataset = get_dataset(horizon, spec, include_lstm_seq=True)
    horizon_steps = duration_steps(spec.horizon, spec.bar_freq)
    returns = get_returns(horizon)
    weight_decay = WEIGHT_DECAY[horizon]

    model_names = [
        "HAR-RV-L", "GARCH(1,1)", "XGBoost (stats)",
        "XGBoost (signature)", "Signature Ridge", "MLP (stats)",
        "MLP (stats+signature)", "Signature LSTM",
    ]
    preds = {name: [] for name in model_names}
    truths = []

    for train_idx, test_idx in walk_forward_folds(len(dataset.y), n_splits=n_splits):
        y_train, y_test = dataset.y[train_idx], dataset.y[test_idx]
        truths.append(y_test)

        har = evaluate_har_log(
            dataset.X_har[train_idx], dataset.y_har_log[train_idx],
            dataset.X_har[test_idx], y_test, horizon_steps,
        )
        preds["HAR-RV-L"].append(har.predictions)

        returns_train = returns.loc[:dataset.times[train_idx[-1]]]
        anchor_times = pd.DatetimeIndex(dataset.times[test_idx])
        g = evaluate_garch(returns_train, returns, anchor_times, y_test, horizon_steps, vol="GARCH")
        preds["GARCH(1,1)"].append(g.predictions)
        # EGARCH temporarily disabled:
        # eg = evaluate_garch(returns_train, returns, anchor_times, y_test, horizon_steps, vol="EGARCH")
        # preds["EGARCH(1,1)"].append(eg.predictions)

        xgb_stats = fit_xgb(dataset.X_stats[train_idx], y_train)
        preds["XGBoost (stats)"].append(np.maximum(xgb_stats.predict(dataset.X_stats[test_idx]), EPS))

        xgb_sig = fit_xgb(dataset.X_sig[train_idx], y_train)
        preds["XGBoost (signature)"].append(np.maximum(xgb_sig.predict(dataset.X_sig[test_idx]), EPS))

        ridge = fit_signature_ridge(dataset.X_sig[train_idx], y_train)
        preds["Signature Ridge"].append(np.maximum(ridge.predict(dataset.X_sig[test_idx]), EPS))

        mlp = fit_mlp(dataset.X_stats[train_idx], y_train, weight_decay=weight_decay)
        preds["MLP (stats)"].append(np.maximum(mlp.predict(dataset.X_stats[test_idx]), EPS))

        X_stats_sig_train = np.hstack([dataset.X_stats[train_idx], dataset.X_sig[train_idx]])
        X_stats_sig_test = np.hstack([dataset.X_stats[test_idx], dataset.X_sig[test_idx]])
        mlp_sig = fit_mlp(X_stats_sig_train, y_train, weight_decay=weight_decay)
        preds["MLP (stats+signature)"].append(np.maximum(mlp_sig.predict(X_stats_sig_test), EPS))

        lstm = fit_signature_lstm(dataset.X_seq[train_idx], y_train, weight_decay=weight_decay)
        preds["Signature LSTM"].append(np.maximum(lstm.predict(dataset.X_seq[test_idx]), EPS))

    y_true = np.concatenate(truths)
    results = {name: WalkForwardResult(name, y_true, np.concatenate(p)) for name, p in preds.items()}
    return compare_models(results, horizon_steps)


full_comparison_tables = {}
for horizon in DAILY_HORIZONS:
    table = run_full_comparison(horizon, dimension_df)
    full_comparison_tables[horizon] = table
    display(
        table.style
        .format({"RMSE": "{:.6g}", "DM stat vs best": "{:.3f}", "DM p-value": "{:.3f}"})
        .apply(lambda col: ["font-weight: bold" if b else "" for b in table["Best"]], subset=["Model", "RMSE"])
        .set_caption(f"{HORIZON_LABELS[horizon]} -- best model is bolded; DM p-value < 0.05 means significantly worse than best")
    )

    # This horizon's dataset (X_har/X_stats/X_sig/X_seq together) is never
    # reused once its table is printed -- drop it now rather than let it sit
    # alongside the next horizon's dataset for the rest of the notebook.
    DATASET_CACHE.clear()
    gc.collect()


,Model,RMSE,Best,DM stat vs best,DM p-value
0,Signature Ridge,0.0129787,True,nan,nan
1,HAR-RV-L,0.013838,False,4.151,0.000
2,XGBoost (signature),0.0171172,False,6.406,0.000
3,"GARCH(1,1)",0.0173405,False,18.298,0.000
4,MLP (stats+signature),0.017461,False,7.367,0.000
5,Signature LSTM,0.017485,False,5.566,0.000
6,MLP (stats),0.0195831,False,8.732,0.000
7,XGBoost (stats),0.0201582,False,9.468,0.000


,Model,RMSE,Best,DM stat vs best,DM p-value
0,Signature Ridge,0.0244977,True,nan,nan
1,HAR-RV-L,0.0294278,False,6.392,0.000
2,Signature LSTM,0.0333818,False,4.429,0.000
3,"GARCH(1,1)",0.0378232,False,11.680,0.000
4,XGBoost (signature),0.0395639,False,4.815,0.000
5,MLP (stats+signature),0.0484486,False,4.650,0.000
6,MLP (stats),0.0502952,False,7.435,0.000
7,XGBoost (stats),0.0605379,False,6.095,0.000


,Model,RMSE,Best,DM stat vs best,DM p-value
0,Signature Ridge,0.0520262,True,nan,nan
1,HAR-RV-L,0.0582481,False,2.246,0.025
2,Signature LSTM,0.0601621,False,2.039,0.041
3,XGBoost (signature),0.0692704,False,3.363,0.001
4,"GARCH(1,1)",0.0906265,False,8.024,0.000
5,MLP (stats+signature),0.094439,False,4.820,0.000
6,XGBoost (stats),0.0948761,False,5.551,0.000
7,MLP (stats),0.104214,False,5.807,0.000


## 9. Combined results

Combines the signature-ablation experiments (1-4) into one tidy DataFrame. Experiment 5's per-horizon tables above are the "which model wins" answer -- this section is just the ablation detail behind Experiment 4's dimension choice.


In [11]:
all_results = pd.concat(
    [
        baseline_df,
        #depth_df,
        augmentation_df,
        dimension_df,
    ],
    ignore_index=True,
)

all_results = ordered_results(all_results)

display(
    all_results[["Horizon", "Experiment", "Setting", "Model", "RMSE"]]
    .head(20)
    .style
    .format({"RMSE": "{:.6g}"})
)


,Horizon,Experiment,Setting,Model,RMSE
50,5 min,Dimensions,Price + activity,Signature XGBoost,0.000416018
0,5 min,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (stats),0.000418914
51,5 min,Dimensions,Price,Signature XGBoost,0.000422925
1,5 min,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (signature),0.000424682
20,5 min,Augmentation,Lead-lag + time,Signature XGBoost,0.000424682
52,5 min,Dimensions,Price + activity + flow,Signature XGBoost,0.000424682
53,5 min,Dimensions,Price + activity,Signature Ridge,0.000453377
2,5 min,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.000466614
21,5 min,Augmentation,Lead-lag + time,Signature Ridge,0.000466614
54,5 min,Dimensions,Price + activity + flow,Signature Ridge,0.000466614


In [12]:
df_5min = all_results[all_results["Horizon"] == "5 min"]
df_5min.head()

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE
50,5 min,5m,Dimensions,Price + activity,Signature XGBoost,0.000416
0,5 min,5m,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (stats),0.000419
51,5 min,5m,Dimensions,Price,Signature XGBoost,0.000423
1,5 min,5m,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (signature),0.000425
20,5 min,5m,Augmentation,Lead-lag + time,Signature XGBoost,0.000425


In [13]:
df_1hr = all_results[all_results["Horizon"] == "1 hour"]
df_1hr.head()

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE
4,1 hour,1h,Baseline,Depth 3 | full path | lead-lag + time,HAR-RV-L,0.001315
5,1 hour,1h,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (signature),0.001513
26,1 hour,1h,Augmentation,Lead-lag + time,Signature XGBoost,0.001513
56,1 hour,1h,Dimensions,Price + activity + flow,Signature XGBoost,0.001513
57,1 hour,1h,Dimensions,Price,Signature XGBoost,0.001579


In [14]:
df_1d = all_results[all_results["Horizon"] == "1 day"]
df_1d.head()

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE
8,1 day,1d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.009559
32,1 day,1d,Augmentation,Lead-lag + time,Signature Ridge,0.009559
62,1 day,1d,Dimensions,Price + activity + range,Signature Ridge,0.009559
33,1 day,1d,Augmentation,Time,Signature Ridge,0.009626
63,1 day,1d,Dimensions,Price + activity,Signature Ridge,0.009667


In [15]:
df_7d = all_results[all_results["Horizon"] == "7 day"]
df_7d.head()

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE
38,7 day,7d,Augmentation,Time,Signature Ridge,0.019101
12,7 day,7d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.019154
39,7 day,7d,Augmentation,Lead-lag + time,Signature Ridge,0.019154
68,7 day,7d,Dimensions,Price + activity + range,Signature Ridge,0.019154
40,7 day,7d,Augmentation,None,Signature Ridge,0.019399


In [16]:
df_30d = all_results[all_results["Horizon"] == "30 day"]
df_30d.head()

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE
44,30 day,30d,Augmentation,Time,Signature Ridge,0.039053
74,30 day,30d,Dimensions,Price,Signature XGBoost,0.039554
16,30 day,30d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.039916
45,30 day,30d,Augmentation,Lead-lag + time,Signature Ridge,0.039916
75,30 day,30d,Dimensions,Price + activity + range,Signature Ridge,0.039916


In [17]:
all_results["Model"].unique()

array(['Signature XGBoost', 'XGBoost (stats)', 'XGBoost (signature)',
       'Signature Ridge', 'HAR-RV-L'], dtype=object)

## 10. Save the result tables

This saves the compact/tidy experiment results and Experiment 5's full model-comparison tables, not large feature arrays.


In [18]:
baseline_df.to_csv(OUTPUT_ROOT / "baseline_results.csv", index=False)

augmentation_df.to_csv(OUTPUT_ROOT / "augmentation_ablation_results.csv", index=False)

dimension_df.to_csv(OUTPUT_ROOT / "dimension_ablation_results.csv", index=False)

all_results.to_csv(OUTPUT_ROOT / "all_experiment_results.csv", index=False)

for horizon, table in full_comparison_tables.items():
    table.to_csv(OUTPUT_ROOT / f"full_comparison_{horizon}.csv", index=False)

print("Saved compact result CSVs to:", OUTPUT_ROOT.resolve())


Saved compact result CSVs to: /Users/mihikaghaisas/Desktop/CMU/MLCapstone/Mihika/outputs
